# val_03 — Can pose tracking find licks the lickometer missed?

**Issue:** [dynamic-foraging-processing#96](https://github.com/AllenNeuralDynamics/dynamic-foraging-processing/issues/96),
"QC request for missed licks" — the mouse licks, the video shows it, the lickometer does not
register it. The request is for QC that finds those cases.

**Approach.** A bottom-view camera tracks the tongue with Lightning Pose. A lick is called from
the pose data when the tongue tip comes within a fixed pixel distance of a spout. Comparing those
events to the lickometer gives three categories:

| category | meaning |
|---|---|
| **matched** | tongue reached the spout and the lickometer registered contact |
| **pose-only** | tongue reached the spout, no lickometer contact — the candidate missed lick |
| **lickometer-only** | lickometer registered contact with no tongue detected at the spout |

**pose-only is the quantity issue #96 is asking about.** Everything below is about how well that
category can be measured and how far it can be trusted.

**The central caveat, stated once up front.** A pose-only event is *either* a lick the lickometer
missed *or* something the pose detector got wrong — the tongue approaching without contacting, or
a tracking error. Event timing alone cannot separate those two. §4 shows why: the tongue-to-spout
distance at confirmed licks is a broad distribution, not a clean value, so any single pixel
threshold cuts through the middle of real behavior. §5 quantifies the cost of moving that
threshold; §6 looks at individual events and pulls video for them.

**Contents**
1. Setup
2. The example session
3. What the pose data looks like
4. How close is the tongue at a lickometer lick?
5. What the pixel threshold buys and costs
6. What pose-only events actually are
7. How often this happens, across all sessions
8. Possible QC metrics
9. Limitations

**Data.** The session in issue #96 (`behavior_856239_2026-07-24_12-50-23`) has no pose output, so
it cannot be reproduced here. The processed pose dataset covers 44 sessions / 15 subjects,
2024-05-31 to 2025-07-03. §3-§6 use one session in detail; §7 uses all of them.

**Code Ocean only** — needs the per-session `intermediate_data/` parquets. Locally the notebook
opens and runs clean, printing a skip line per data-dependent cell.

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
    mask_keypoint_data,
)
from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_lickometer_utils import (
    detect_licks,
    filter_timestamps_refractory,
    calculate_metrics_witheventkeys,
    extract_clips_ffmpeg_encode,
)
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import find_session_dir
from aind_dynamic_foraging_behavior_video_analysis.video_alignment import (
    session_time_to_video_time,
)
from aind_dynamic_foraging_behavior_video_analysis.kinematics.video_clip_utils import (
    get_video_time,
    find_labeled_video,
)

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV     = "codeocean"
    SCRATCH = Path("/root/capsule/scratch")
    DATA    = Path("/root/capsule/data")
else:
    ENV     = "local"
    SCRATCH = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local").parent
    DATA    = SCRATCH

IS_CO    = ENV == "codeocean"
FIG_DIR  = SCRATCH / "figures" / "val_03_missed_licks"
SAVE_FIG = False

SESSION_DIR     = SCRATCH / "session_analysis_mlk"
EXAMPLE_SESSION = "behavior_716325_2024-05-31_10-31-14"

# Detection parameters. PIXEL_THRESHOLD and T_REFRACTORY are val_02's chosen operating
# point; MATCH_WINDOW is the coincidence window for calling two events the same lick.
CONF            = 0.8    # keypoint confidence floor for masking
PIXEL_THRESHOLD = 30     # tongue-to-spout distance that counts as a lick, in pixels
T_REFRACTORY    = 0.1    # s; suppresses re-detection from jitter around the threshold
MATCH_WINDOW    = 0.1    # s; pose and lickometer events this close are the same lick

# Colors, fixed across every figure below.
C_MATCH = PALETTE["baseline"]   # matched: pose and lickometer agree
C_POSE  = PALETTE["pos"]        # pose-only: candidate missed lick
C_LICKO = PALETTE["neg"]        # lickometer-only

print("ENV={}".format(ENV))

## 2. The example session

Session `behavior_716325_2024-05-31_10-31-14`, from the per-session `intermediate_data/`
parquets.

**Time bases.** `kps_raw_*.parquet` carries `time`, which is video-relative
(`Behav_Time - Behav_Time[0]`). `tongue_kins.parquet` carries both that `time` and
`time_in_session`, which is go-cue-relative and is the base `nwb_df_licks['timestamps']` uses.
The offset between them is `first_go_cue - first_frame_behavior_time`; `video_clip_utils.get_video_time`
reads it off the first row of `tongue_kins`, and `video_alignment.session_time_to_video_time`
applies it. Analysis runs in session time; §6 converts back to video time to cut clips.

(`video_alignment.compute_video_session_offset` computes the same offset from the raw acquisition
CSV plus the first go-cue time. That path is for when the video CSV is at hand; here the
intermediate tables already carry both bases.)

Positions come from `kps_raw_*`, not from `tongue_kins`: the latter is post-filter and its x/y are
interpolated across gaps, which corrupts values at exactly the protrusion edges where the
threshold is crossed.

The `spout_l` keypoint is the animal's **right** spout and vice versa — the bottom camera mirrors
left/right, and the training labels follow the camera. The swap below is deliberate.

In [ ]:
def load_session(inter):
    """Load one session's tongue keypoints, spout positions and lickometer licks.

    Parameters
    ----------
    inter : pathlib.Path
        The session's ``intermediate_data`` directory.

    Returns
    -------
    dict
        ``tongue`` (masked keypoint table on session time), ``spoutL``/``spoutR`` (mean
        positions, already de-mirrored), ``licks`` (lickometer times, session time) and
        ``video_offset`` (seconds to add to session time to get video time, via
        :func:`video_clip_utils.get_video_time`).
    """
    keypoint_dfs = {
        key: pd.read_parquet(inter / "kps_raw_{}.parquet".format(key))
        for key in ["tongue_tip_center", "spout_l", "spout_r"]
    }
    kins = pd.read_parquet(inter / "tongue_kins.parquet",
                           columns=["time", "time_in_session"])

    tongue = mask_keypoint_data(keypoint_dfs, "tongue_tip_center", confidence_threshold=CONF)
    # kps_raw_* `time` is video-relative. get_video_time reads the session->video offset off
    # the first row of tongue_kins, which carries both bases; calling it at 0 returns the
    # offset itself.
    video_offset = float(get_video_time(0.0, kins))
    tongue["time"] = kins["time_in_session"].values

    return {
        # NB the bottom camera mirrors left/right, so spout_r holds the left spout.
        "spoutL": np.mean(keypoint_dfs["spout_r"][["x", "y"]], 0),
        "spoutR": np.mean(keypoint_dfs["spout_l"][["x", "y"]], 0),
        "tongue": tongue,
        "licks": np.sort(
            pd.read_parquet(inter / "nwb_df_licks.parquet")["timestamps"].to_numpy()),
        "video_offset": video_offset,
    }


def add_spout_distance(tongue, spoutL, spoutR):
    """Add per-frame distance to the nearer spout. Untracked frames stay NaN."""
    tongue = tongue.copy()
    tracked = tongue[["x", "y"]].notna().all(axis=1)
    xy = tongue.loc[tracked, ["x", "y"]].to_numpy()
    dL = np.linalg.norm(xy - np.array([spoutL["x"], spoutL["y"]]), axis=1)
    dR = np.linalg.norm(xy - np.array([spoutR["x"], spoutR["y"]]), axis=1)
    tongue["spout_distance"] = np.nan
    tongue.loc[tracked, "spout_distance"] = np.minimum(dL, dR)
    return tongue

In [ ]:
if not IS_CO:
    print("[skip] Code Ocean only: needs session_analysis_mlk/<session>/intermediate_data/.")
else:
    inter = find_session_dir(EXAMPLE_SESSION, roots=[SESSION_DIR]) / "intermediate_data"
    S = load_session(inter)
    S["tongue"] = add_spout_distance(S["tongue"], S["spoutL"], S["spoutR"])

    tongue = S["tongue"]
    all_licks = S["licks"]
    VIDEO_OFFSET = S["video_offset"]

    tracked = tongue["x"].notna()
    duration = float(tongue["time"].max() - tongue["time"].min())

    print("Session:          {}".format(EXAMPLE_SESSION))
    print("Frames:           {:,} over {:.0f} s".format(len(tongue), duration))
    print("Tracked at conf >= {}: {:,} ({:.0%})".format(CONF, int(tracked.sum()),
                                                        tracked.mean()))
    print("Lickometer licks: {:,}  ({:.2f} / s)".format(len(all_licks),
                                                        len(all_licks) / duration))
    print("Video offset:     {:.3f} s  (session time + this = video time)".format(VIDEO_OFFSET))

## 3. What the pose data looks like

Two panels, both from the same 8-second window of ordinary licking.

**Top** — tongue tip position in the camera frame, with the two spout positions marked. Each
excursion toward a spout is a protrusion.

**Bottom** — distance from the tongue tip to the nearer spout over time. Gaps are frames where
the keypoint confidence fell below 0.8, which is what happens when the tongue is inside the
mouth and there is nothing to track. The dashed line is the 30 px detection threshold; the
lickometer's own lick times are the ticks along the bottom.

This is the raw material for everything that follows: a distance trace that dips toward zero on
each protrusion, and a set of lickometer events that should line up with those dips.

In [ ]:
if not IS_CO:
    print("[skip] Figure 1 needs the example session.")
else:
    # An 8 s window starting at the 20th lickometer lick, so it lands in real licking.
    w0 = float(all_licks[20]) - 1.0
    w1 = w0 + 8.0
    win = tongue[(tongue["time"] >= w0) & (tongue["time"] <= w1)]
    win_licks = all_licks[(all_licks >= w0) & (all_licks <= w1)]

    fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                            gridspec_kw={"height_ratios": [1.3, 1]})

    # -- top: tongue x and y, with spout positions --
    axs[0].plot(win["time"], win["x"], lw=1.2, color=PALETTE["all"], label="tongue x")
    axs[0].plot(win["time"], win["y"], lw=1.2, color=PALETTE["accent"], label="tongue y")
    for spout, name in [(S["spoutL"], "left spout"), (S["spoutR"], "right spout")]:
        axs[0].axhline(spout["y"], color=PALETTE["neutral"], ls=":", lw=1)
        axs[0].text(w1, spout["y"], "  " + name, va="center", fontsize=8,
                    color=PALETTE["neutral"])
    axs[0].set_ylabel("Position (px)")
    axs[0].legend(loc="upper left", fontsize=8, ncol=2)
    axs[0].set_title("Tongue tip, {:.0f}-{:.0f} s".format(w0, w1))

    # -- bottom: distance to nearer spout --
    axs[1].plot(win["time"], win["spout_distance"], lw=1.4, color=PALETTE["all"])
    axs[1].axhline(PIXEL_THRESHOLD, color=PALETTE["neutral"], ls="--", lw=1,
                   label="{} px threshold".format(PIXEL_THRESHOLD))
    axs[1].plot(win_licks, np.full(len(win_licks), -4), "|", ms=12, mew=2,
                color=C_LICKO, label="lickometer lick")
    axs[1].set_ylim(-8, 120)
    axs[1].set_ylabel("Distance to nearer\nspout (px)")
    axs[1].set_xlabel("Time in session (s)")
    axs[1].legend(loc="upper right", fontsize=8)

    for ax in axs:
        style_ax(ax)
    fig.suptitle("Pose tracking of the tongue during licking", y=0.98)
    fig.tight_layout()
    save_fig(fig, "fig1_pose_example", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    print("Window: {:.1f}-{:.1f} s | frames {:,} | tracked {:.0%} | lickometer licks {}".format(
        w0, w1, len(win), win["x"].notna().mean(), len(win_licks)))

### What to take from it

The distance trace is only defined while the tongue is visible, so it is a series of short
tracked excursions separated by untracked gaps, not a continuous signal. Detection is therefore a
question about each excursion: did it get close enough to the spout to count?

## 4. How close is the tongue at a lickometer lick?

Take every lickometer lick and ask how near the tongue tip got to the spout around that moment —
the minimum distance within ±100 ms of the lick. If contact were cleanly separable in the pose
data, these values would cluster tightly at small distances.

**Everything here uses confidence-filtered positions.** `spout_distance` is defined only on frames
that survived masking at confidence ≥ `CONF`, and both distributions below are measured from the
same filtered arrays. A window containing no surviving frame yields no value and is dropped.

The comparison distribution is the same measurement at times drawn uniformly through the session.
Because it is measured from the same filtered arrays, it is conditioned on the tongue having been
*visible* at a random moment — and the tongue is generally only visible when protruded. So it is
not "wherever the tongue happens to be"; it is "wherever a tracked tongue happens to be". The two
window-coverage rates are printed below, and they differ a lot. The histograms are plotted as
densities for that reason: the two samples have different n.

In [ ]:
def min_distance_near(times, dist, event_times, halfwidth):
    """Minimum tracked tongue-spout distance within +/- halfwidth of each event.

    Parameters
    ----------
    times, dist : ndarray
        Tracked frames only, sorted by time, no NaN.
    event_times : ndarray
        Times to measure around.
    halfwidth : float
        Half-width of the search window, in seconds.

    Returns
    -------
    ndarray
        One value per event; NaN where no tracked frame falls in the window.
    """
    event_times = np.asarray(event_times, dtype=float)
    lo = np.searchsorted(times, event_times - halfwidth, side="left")
    hi = np.searchsorted(times, event_times + halfwidth, side="right")
    out = np.full(event_times.size, np.nan)
    for k in range(event_times.size):
        if hi[k] > lo[k]:
            out[k] = dist[lo[k]:hi[k]].min()
    return out

In [ ]:
if not IS_CO:
    print("[skip] Figure 2 needs the example session.")
else:
    # Confidence-filtered frames only: spout_distance is NaN wherever masking removed x/y.
    trk = tongue.loc[tongue["spout_distance"].notna(), ["time", "spout_distance"]]
    trk_t = trk["time"].to_numpy()
    trk_d = trk["spout_distance"].to_numpy()

    HALFWIDTH = 0.1
    d_at_lick = min_distance_near(trk_t, trk_d, all_licks, HALFWIDTH)

    # Same filtered arrays, so the null is conditioned on the tongue being tracked.
    rng = np.random.default_rng(0)
    random_times = rng.uniform(tongue["time"].min(), tongue["time"].max(), len(all_licks))
    d_at_random = min_distance_near(trk_t, trk_d, random_times, HALFWIDTH)

    no_track = np.isnan(d_at_lick)
    no_track_random = np.isnan(d_at_random)

    fig, axs = plt.subplots(1, 2, figsize=(11, 4))

    # Densities, not counts: the two samples have different n after dropping empty windows.
    bins = np.arange(0, 121, 3)
    axs[0].hist(d_at_lick[~no_track], bins=bins, density=True, color=C_LICKO, alpha=0.75,
                label="at lickometer licks (n={:,})".format(int((~no_track).sum())))
    axs[0].hist(d_at_random[~no_track_random], bins=bins, density=True, histtype="step", lw=1.6,
                color=PALETTE["neutral"],
                label="at random tracked times (n={:,})".format(int((~no_track_random).sum())))
    axs[0].axvline(PIXEL_THRESHOLD, color=PALETTE["all"], ls="--", lw=1.2)
    axs[0].text(PIXEL_THRESHOLD + 2, axs[0].get_ylim()[1] * 0.92,
                "{} px".format(PIXEL_THRESHOLD), fontsize=8)
    axs[0].set_xlabel("Min distance to spout within +/-{:.0f} ms (px)".format(HALFWIDTH * 1000))
    axs[0].set_ylabel("Density")
    axs[0].set_title("Distance at lickometer licks")
    axs[0].legend(fontsize=8)

    # CDF, with the candidate thresholds marked
    srt = np.sort(d_at_lick[~no_track])
    cdf = np.arange(1, srt.size + 1) / srt.size
    axs[1].plot(srt, cdf, lw=2, color=C_LICKO)
    for px in [20, 30, 40, 50]:
        frac = float((srt <= px).mean())
        axs[1].plot([px, px], [0, frac], color=PALETTE["neutral"], ls=":", lw=1)
        axs[1].plot([0, px], [frac, frac], color=PALETTE["neutral"], ls=":", lw=1)
        axs[1].annotate("{} px -> {:.0%}".format(px, frac), (px, frac),
                        textcoords="offset points", xytext=(6, -10), fontsize=8)
    axs[1].set_xlim(0, 120)
    axs[1].set_ylim(0, 1)
    axs[1].set_xlabel("Min distance to spout (px)")
    axs[1].set_ylabel("Fraction of lickometer licks at or below")
    axs[1].set_title("How much a threshold captures")

    for ax in axs:
        style_ax(ax)
    fig.tight_layout()
    save_fig(fig, "fig2_distance_at_licks", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    pct = np.nanpercentile(d_at_lick, [10, 25, 50, 75, 90])
    print("Distance at lickometer licks (px), confidence >= {}:".format(CONF))
    print("  10th {:.1f} | 25th {:.1f} | median {:.1f} | 75th {:.1f} | 90th {:.1f}".format(*pct))
    print("\nWindows with no tracked frame (dropped from the histogram):")
    print("  at lickometer licks:    {:>6,} / {:,}  ({:.1%})".format(
        int(no_track.sum()), no_track.size, no_track.mean()))
    print("  at random times:        {:>6,} / {:,}  ({:.1%})".format(
        int(no_track_random.sum()), no_track_random.size, no_track_random.mean()))
    print("  frames passing confidence >= {}: {:.1%} of the session".format(
        CONF, float(tongue["x"].notna().mean())))

### What to take from it

**The distance at a confirmed lick is a broad distribution, not a value.** The lickometer says
contact happened; the pose data puts the tongue anywhere across tens of pixels at those same
moments. Some of that spread is real — the tongue contacts different parts of the spout — and some
is tracking error and the ±100 ms window.

Three consequences:

1. **No threshold is correct.** The printed percentiles say directly how much of real licking any
   given choice captures. A threshold tight enough to exclude non-contact approaches also discards
   a substantial share of confirmed licks; the ones it discards get counted as lickometer-only.
2. **A fraction of lickometer licks have no tracked tongue at all** in the window. Those cannot be
   confirmed or denied from pose — the tongue was simply not visible at confidence ≥ `CONF`. They
   are a floor on how well any pose-based QC can do, and they are reported above.
3. **The null is conditioned, and conservatively so.** Far more random windows than lick windows
   contain no tracked frame, because the tongue is usually inside the mouth. The random
   distribution therefore describes moments when the tongue was already out — the hardest case to
   distinguish from a lick. The separation that survives that conditioning is real separation;
   a null drawn over all frames regardless of tracking would look more separated and mean less.

That separation is what makes the approach viable at all: the tongue really is closer to the spout
at lick times than at arbitrary tracked times. The overlap is what makes it imprecise.

## 5. What the pixel threshold buys and costs

Sweep the detection threshold and, at each value, classify every event into the three categories
from the top of the notebook.

The quantity the issue cares about is the **pose-only count** — how many candidate missed licks
the method reports. The quantity that limits trust in it is **precision**: of the licks the pose
detector calls, what fraction the lickometer also registered.

These move in opposite directions, and that trade is the whole story of this section.

In [ ]:
def classify_events(tongue, licks, spoutL, spoutR, threshold,
                    t_refractory=None, match_window=None):
    """Detect pose licks at one threshold and classify against the lickometer.

    Returns
    -------
    dict
        Counts (``n_matched``, ``n_pose_only``, ``n_lickometer_only``), the derived
        fractions, and the pose-only / lickometer-only event times.
    """
    t_refractory = T_REFRACTORY if t_refractory is None else t_refractory
    match_window = MATCH_WINDOW if match_window is None else match_window

    pose_licks = detect_licks(tongue, spoutL, spoutR, threshold)
    pose_licks = filter_timestamps_refractory(pose_licks, t_refractory)
    licko_licks = filter_timestamps_refractory(list(licks), t_refractory)

    # Pose events are passed first, so the second count is unmatched lickometer events
    # and the third is unmatched pose events (see val_02 for the argument order).
    n_matched, n_lickometer_only, n_pose_only, pose_tbl, licko_tbl = (
        calculate_metrics_witheventkeys(pose_licks, licko_licks, time_window=match_window))

    n_pose = n_matched + n_pose_only
    n_licko = n_matched + n_lickometer_only
    return {
        "threshold": threshold,
        "n_matched": n_matched,
        "n_pose_only": n_pose_only,
        "n_lickometer_only": n_lickometer_only,
        "n_pose": n_pose,
        "n_lickometer": n_licko,
        # Precision: of the licks pose calls, the fraction the lickometer confirms.
        "precision": n_matched / n_pose if n_pose > 0 else np.nan,
        # Recall: of the lickometer's licks, the fraction pose also finds.
        "recall": n_matched / n_licko if n_licko > 0 else np.nan,
        "pose_only_times": pose_tbl.loc[
            pose_tbl["Status"] == "False Negative", "Time"].to_numpy(),
        "lickometer_only_times": licko_tbl.loc[
            licko_tbl["Status"] == "False Positive", "Time"].to_numpy(),
    }

In [ ]:
if not IS_CO:
    print("[skip] Figure 3 needs the example session.")
else:
    sweep_thresholds = np.arange(10, 61, 5)
    sweep = [classify_events(tongue, all_licks, S["spoutL"], S["spoutR"], px)
             for px in sweep_thresholds]
    sweep_df = pd.DataFrame([{k: v for k, v in r.items() if not k.endswith("_times")}
                             for r in sweep])
    sweep_df["pose_only_per_min"] = sweep_df["n_pose_only"] / (duration / 60.0)
    display(sweep_df[["threshold", "n_matched", "n_pose_only", "n_lickometer_only",
                      "precision", "recall", "pose_only_per_min"]].round(3))

In [ ]:
if not IS_CO:
    print("[skip] Figure 3 needs the example session.")
else:
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))

    # -- panel 1: precision and recall --
    axs[0].plot(sweep_df["threshold"], sweep_df["precision"], "o-", color=C_POSE,
                label="precision: of pose licks,\nfraction lickometer confirms")
    axs[0].plot(sweep_df["threshold"], sweep_df["recall"], "s-", color=C_LICKO,
                label="recall: of lickometer licks,\nfraction pose finds")
    axs[0].axvline(PIXEL_THRESHOLD, color=PALETTE["neutral"], ls="--", lw=1)
    axs[0].set_ylim(0, 1.02)
    axs[0].set_xlabel("Pixel threshold")
    axs[0].set_ylabel("Fraction")
    axs[0].set_title("Agreement vs threshold")
    axs[0].legend(fontsize=7, loc="lower center")

    # -- panel 2: the three counts --
    axs[1].plot(sweep_df["threshold"], sweep_df["n_matched"], "o-", color=C_MATCH,
                label="matched")
    axs[1].plot(sweep_df["threshold"], sweep_df["n_pose_only"], "o-", color=C_POSE,
                label="pose-only (candidate missed)")
    axs[1].plot(sweep_df["threshold"], sweep_df["n_lickometer_only"], "o-", color=C_LICKO,
                label="lickometer-only")
    axs[1].axvline(PIXEL_THRESHOLD, color=PALETTE["neutral"], ls="--", lw=1)
    axs[1].set_xlabel("Pixel threshold")
    axs[1].set_ylabel("Events")
    axs[1].set_title("Event counts vs threshold")
    axs[1].legend(fontsize=8)

    # -- panel 3: the trade, as a curve --
    axs[2].plot(sweep_df["precision"], sweep_df["n_pose_only"], "-", color=PALETTE["neutral"],
                lw=1, zorder=1)
    sc = axs[2].scatter(sweep_df["precision"], sweep_df["n_pose_only"],
                        c=sweep_df["threshold"], cmap="viridis", s=55, zorder=2)
    for _, r in sweep_df.iterrows():
        axs[2].annotate("{:.0f}".format(r["threshold"]), (r["precision"], r["n_pose_only"]),
                        textcoords="offset points", xytext=(6, 3), fontsize=7)
    fig.colorbar(sc, ax=axs[2], label="Pixel threshold")
    axs[2].set_xlabel("Precision")
    axs[2].set_ylabel("Pose-only events reported")
    axs[2].set_title("The trade")

    for ax in axs:
        style_ax(ax)
    fig.suptitle("Pixel threshold controls how many missed licks you find "
                 "and how many you can trust", y=1.02)
    fig.tight_layout()
    save_fig(fig, "fig3_threshold_tradeoff", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

### What to take from it

Raising the threshold makes the pose detector fire on more excursions. That finds more candidate
missed licks and simultaneously lowers precision, because the extra detections include approaches
that never made contact. Lowering it does the reverse: the events that survive are more likely to
be real licks, but there are fewer of them and more genuine licks are missed entirely.

**The third panel is the operating-point choice, laid out.** Each point is one threshold; moving
right buys confidence in the reported events, moving up buys more of them. There is no point that
does both, and nothing in this data picks one for you — the choice depends on whether the QC is
meant to flag sessions for review (favor recall) or to produce a list of events someone will act
on (favor precision).

Note also that pose-only counts never reach zero at any threshold. Some of that is real missed
licks; some is the detector firing on non-contact approaches. §6 is where that gets examined
directly.

## 6. What pose-only events actually are

Counts cannot settle whether a pose-only event is a missed lick or a detector error. Two ways to
look:

1. **Trajectory panels** — tongue x, y and distance-to-spout around each event. A real lick shows
   a protrusion that reaches the spout and dwells; a detector error shows a shallow or glancing
   approach, or a discontinuity that indicates a tracking failure.
2. **Video clips** — the labeled video cut around each event, which is the only direct evidence.

The analysis above runs on go-cue-relative session time and the video starts at its own zero, so
clip times go back through `session_time_to_video_time` with the offset from §2.
`find_labeled_video` locates the session's labeled video under the data root.

In [ ]:
def plot_event_panels(tongue, event_times, event_label, clip_length=1.0, n_max=4,
                      threshold=None):
    """Tongue x, y and spout distance around each of up to n_max events."""
    threshold = PIXEL_THRESHOLD if threshold is None else threshold
    event_times = np.asarray(event_times, dtype=float)[:n_max]
    if event_times.size == 0:
        print("No events to plot.")
        return

    fig, axs = plt.subplots(3, event_times.size, figsize=(3.1 * event_times.size, 6),
                            sharex="col", squeeze=False)
    for j, t0 in enumerate(event_times):
        win = tongue[(tongue["time"] >= t0 - clip_length / 2) &
                     (tongue["time"] <= t0 + clip_length / 2)]
        for row, (col, label) in enumerate([("y", "Y (px)"), ("x", "X (px)"),
                                            ("spout_distance", "Dist. to spout (px)")]):
            ax = axs[row][j]
            ax.plot(win["time"], win[col], ".-", ms=2, lw=1,
                    color=PALETTE["all"] if row < 2 else C_POSE)
            ax.axvline(t0, color=PALETTE["neutral"], ls="--", lw=1)
            if row == 2:
                ax.axhline(threshold, color=PALETTE["neutral"], ls=":", lw=1)
                ax.set_ylim(0, 120)
                ax.set_xlabel("Time in session (s)")
            if j == 0:
                ax.set_ylabel(label)
            style_ax(ax)
        axs[0][j].set_title("{:.2f} s".format(t0), fontsize=9)

    fig.suptitle("{} (dashed = event, dotted = {} px threshold)".format(event_label, threshold),
                 y=1.01)
    fig.tight_layout()
    return fig

In [ ]:
if not IS_CO:
    print("[skip] Event panels need the example session.")
else:
    ref = classify_events(tongue, all_licks, S["spoutL"], S["spoutR"], PIXEL_THRESHOLD)
    pose_only_times = ref["pose_only_times"]
    lickometer_only_times = ref["lickometer_only_times"]

    print("At {} px / {} s refractory / {} s match window:".format(
        PIXEL_THRESHOLD, T_REFRACTORY, MATCH_WINDOW))
    print("  matched:          {:,}".format(ref["n_matched"]))
    print("  pose-only:        {:,}   <- candidate missed licks".format(ref["n_pose_only"]))
    print("  lickometer-only:  {:,}".format(ref["n_lickometer_only"]))
    print("  precision:        {:.3f}".format(ref["precision"]))
    print("  recall:           {:.3f}".format(ref["recall"]))

    fig = plot_event_panels(tongue, pose_only_times, "Pose-only events (candidate missed licks)")
    if fig is not None:
        save_fig(fig, "fig4_pose_only_panels", fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()

In [ ]:
if not IS_CO:
    print("[skip] Comparison panels need the example session.")
else:
    # Matched licks, for contrast: this is what a confirmed lick looks like.
    pose_all = detect_licks(tongue, S["spoutL"], S["spoutR"], PIXEL_THRESHOLD)
    pose_all = np.asarray(filter_timestamps_refractory(pose_all, T_REFRACTORY))
    matched_times = np.setdiff1d(pose_all, pose_only_times)

    fig = plot_event_panels(tongue, matched_times, "Matched licks, for comparison")
    if fig is not None:
        save_fig(fig, "fig5_matched_panels", fig_dir=FIG_DIR, save=SAVE_FIG)
        plt.show()

In [ ]:
if not IS_CO:
    print("[skip] Clip extraction needs the example session and the labeled video.")
else:
    CLIP_LENGTH = 1.0
    N_CLIPS = 5

    try:
        labeled_video = find_labeled_video(EXAMPLE_SESSION, DATA)
    except FileNotFoundError as exc:
        labeled_video = None
        print("[skip] {}".format(exc))
        print("       find_labeled_video looks for"
              " <data_root>/<session>*/pred_outputs/video_preds/labeled_videos/*_labeled.mp4")

    if labeled_video is not None:
        print("Labeled video: {}".format(labeled_video))
        for times, name in [(pose_only_times, "pose_only"),
                            (lickometer_only_times, "lickometer_only")]:
            if len(times) == 0:
                print("{}: no events".format(name))
                continue
            # Session time -> video time, then back up half a clip to center the event.
            starts = (session_time_to_video_time(
                np.asarray(times[:N_CLIPS], dtype=float), VIDEO_OFFSET) - CLIP_LENGTH / 2)
            out_dir = SCRATCH / "labeled_clips" / "val_03" / name
            print("{}: {} clips -> {}".format(name, len(starts), out_dir))
            # extract_clips_ffmpeg_encode re-encodes, so the cut lands on the requested frame.
            # video_clip_utils.extract_clips_ffmpeg_after_reencode is faster but stream-copies,
            # which snaps the start to the nearest preceding keyframe -- too coarse for a 1 s
            # clip centered on a single event.
            extract_clips_ffmpeg_encode(str(labeled_video), starts, CLIP_LENGTH, str(out_dir))

### What to take from it

Compare the pose-only panels against the matched panels directly. Events whose distance trace
dips well below the threshold and dwells there look like licks the lickometer failed to register.
Events that only graze the threshold, or whose trace jumps discontinuously, are the detector
firing on something else.

**The clips are the ground truth, and they need to be watched.** Nothing in this notebook can
decide the question on its own — that is the honest limit of the method, and it is why §8 proposes
QC that flags candidates for review rather than QC that silently corrects the lick record.

## 7. How often this happens, across all sessions

One session cannot say whether missed licks are a widespread problem or a property of one
recording. This runs the §5 classification at the fixed operating point over every session with
per-session pose output, and reports the spread.

Tracked fraction is carried alongside every session, because a session where the tongue is rarely
visible will produce few pose detections for reasons that have nothing to do with the lickometer.

In [ ]:
def process_session_dir(sdir, threshold=None):
    """Run the §5 classification on one session directory. Returns None if unusable."""
    threshold = PIXEL_THRESHOLD if threshold is None else threshold
    inter = Path(sdir) / "intermediate_data"
    needed = ["kps_raw_tongue_tip_center.parquet", "kps_raw_spout_l.parquet",
              "kps_raw_spout_r.parquet", "tongue_kins.parquet", "nwb_df_licks.parquet"]
    if not all((inter / f).exists() for f in needed):
        return None

    sess = load_session(inter)
    sess["tongue"] = add_spout_distance(sess["tongue"], sess["spoutL"], sess["spoutR"])
    tg = sess["tongue"]
    dur = float(tg["time"].max() - tg["time"].min())
    if dur <= 0 or len(sess["licks"]) == 0:
        return None

    res = classify_events(tg, sess["licks"], sess["spoutL"], sess["spoutR"], threshold)
    return {
        "session": Path(sdir).name,
        "duration_s": dur,
        "tracked_frac": float(tg["x"].notna().mean()),
        "n_matched": res["n_matched"],
        "n_pose_only": res["n_pose_only"],
        "n_lickometer_only": res["n_lickometer_only"],
        "n_lickometer": res["n_lickometer"],
        "precision": res["precision"],
        "recall": res["recall"],
        "pose_only_per_min": res["n_pose_only"] / (dur / 60.0),
        # Share of the lickometer record that pose says is missing a partner event.
        "pose_only_frac": res["n_pose_only"] / res["n_pose"] if res["n_pose"] > 0 else np.nan,
    }

In [ ]:
MAX_SESSIONS = None   # set to an int to cut the loop short while iterating

if not IS_CO:
    print("[skip] Multi-session sweep needs session_analysis_mlk/*/intermediate_data/.")
else:
    session_dirs = sorted(glob.glob(str(SESSION_DIR / "*")))
    if MAX_SESSIONS is not None:
        session_dirs = session_dirs[:MAX_SESSIONS]

    rows = []
    for k, sdir in enumerate(session_dirs, 1):
        try:
            row = process_session_dir(sdir)
        except Exception as exc:
            print("  [{}/{}] {}: FAILED ({})".format(k, len(session_dirs),
                                                     Path(sdir).name, exc))
            continue
        if row is None:
            continue
        rows.append(row)
        print("  [{}/{}] {}  pose-only {:>4}  precision {:.2f}  tracked {:.0%}".format(
            k, len(session_dirs), row["session"], row["n_pose_only"],
            row["precision"], row["tracked_frac"]))

    sessions_df = pd.DataFrame(rows)
    print("\nSessions analyzed: {}".format(len(sessions_df)))

In [ ]:
if not IS_CO or len(sessions_df) == 0:
    print("[skip] Figure 6 needs the multi-session sweep.")
else:
    order = sessions_df.sort_values("pose_only_per_min").reset_index(drop=True)

    fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))

    # -- panel 1: per-session rate, ranked --
    sc = axs[0].scatter(range(len(order)), order["pose_only_per_min"],
                        c=order["tracked_frac"], cmap="viridis", s=42)
    med = order["pose_only_per_min"].median()
    axs[0].axhline(med, color=PALETTE["neutral"], ls="--", lw=1,
                   label="median {:.2f}/min".format(med))
    fig.colorbar(sc, ax=axs[0], label="Tracked fraction")
    axs[0].set_xlabel("Session (ranked)")
    axs[0].set_ylabel("Pose-only events per minute")
    axs[0].set_title("Candidate missed licks per session")
    axs[0].legend(fontsize=8)

    # -- panel 2: is a high rate just poor tracking? --
    axs[1].scatter(sessions_df["tracked_frac"], sessions_df["pose_only_frac"],
                   color=C_POSE, s=42)
    axs[1].set_xlabel("Tracked fraction")
    axs[1].set_ylabel("Pose-only / all pose licks")
    axs[1].set_title("Rate vs tracking quality")

    # -- panel 3: precision spread --
    axs[2].hist(sessions_df["precision"].dropna(), bins=np.arange(0, 1.05, 0.05),
                color=C_MATCH, alpha=0.85)
    axs[2].set_xlabel("Precision (of pose licks, fraction lickometer confirms)")
    axs[2].set_ylabel("Sessions")
    axs[2].set_title("Precision across sessions")

    for ax in axs:
        style_ax(ax)
    fig.suptitle("Across {} sessions at {} px".format(len(sessions_df), PIXEL_THRESHOLD),
                 y=1.02)
    fig.tight_layout()
    save_fig(fig, "fig6_across_sessions", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

    display(sessions_df[["session", "duration_s", "tracked_frac", "n_lickometer",
                         "n_pose_only", "precision", "pose_only_per_min"]]
            .sort_values("pose_only_per_min", ascending=False).round(3).head(12))

### What to take from it

Panel 1 gives the spread of candidate missed licks per minute. A QC threshold, if one is wanted,
would be a cut on this axis — and the shape of the distribution says whether that cut isolates a
few outlier sessions or slices through a continuum.

Panel 2 is the control that has to be checked before believing panel 1. If pose-only rate tracked
tightly with tracking quality, the metric would mostly be measuring the pose model, not the
lickometer.

Panel 3 shows how much the precision of the approach itself varies session to session, which
bounds how uniformly any fixed threshold will behave.

## 8. Possible QC metrics

Characterizations, not recommendations — each is computable from what is above, and each has a
stated failure mode. The false-positive behavior in §5 is what makes the last two worth having
alongside the first.

### A. Session-level rate
**`pose_only_per_min`** — candidate missed licks per minute, from §7.

Flags a whole session for review. Cheap, needs no trial alignment. Interpretable only *relative to
the cohort*, since its absolute value depends on the pixel threshold; it should be read against
the distribution in §7 panel 1, not against a number fixed in advance. Must be reported next to
`tracked_frac`, or a poorly tracked session looks like a lickometer problem.

### B. Per-trial flag
A trial where the pose detector places a lick inside the response window but the lickometer
records none. Points at specific trials, which is what someone reviewing data actually wants, and
is the form closest to what issue #96 describes. Sensitive to trial-window choice, and inherits
the full pose false-positive rate — a flagged trial is a candidate, not a finding.

### C. High-confidence subset, using the §4 distribution
The §5 precision is the cost of counting every threshold crossing equally. §4 gives a way to
weight them: keep only pose-only events whose minimum distance to the spout falls below a low
percentile of the distance distribution *at confirmed lickometer licks*. Those events are ones the
tongue reached as convincingly as a known lick. Fewer events, higher trust — the right input to a
manual review queue.

### D. Coverage guard
`tracked_frac`, and the fraction of lickometer licks with no tracked tongue nearby (§4). Not a
missed-lick metric — a gate. Below some coverage the other three are not interpretable, and the
honest QC output is "cannot assess" rather than a number.

In [ ]:
RESPONSE_WINDOW = (0.0, 2.0)   # s after go cue; matches the cue-response window used elsewhere
CONFIDENT_PCTL  = 25           # percentile of confirmed-lick distance for metric C

if not IS_CO:
    print("[skip] Metric demonstration needs the example session.")
else:
    # --- C: high-confidence pose-only events ---
    d_pose_only = min_distance_near(trk_t, trk_d, pose_only_times, 0.05)
    cutoff = float(np.nanpercentile(d_at_lick, CONFIDENT_PCTL))
    confident = pose_only_times[np.nan_to_num(d_pose_only, nan=np.inf) <= cutoff]
    print("C. high-confidence subset")
    print("   cutoff = {}th pctl of distance at confirmed licks = {:.1f} px".format(
        CONFIDENT_PCTL, cutoff))
    print("   pose-only events: {:,} -> high-confidence: {:,} ({:.0%})".format(
        len(pose_only_times), len(confident),
        len(confident) / max(len(pose_only_times), 1)))

    # --- B: per-trial flag ---
    trials_path = inter / "nwb_df_trials.parquet"
    if not trials_path.exists():
        print("\nB. [skip] nwb_df_trials.parquet not found")
    else:
        trials = pd.read_parquet(trials_path)
        cue_col = "goCue_start_time_in_session"
        if cue_col not in trials.columns:
            print("\nB. [skip] {} not in nwb_df_trials ({})".format(
                cue_col, list(trials.columns)[:6]))
        else:
            cues = trials[cue_col].dropna().to_numpy()
            def _n_in_window(times, cue):
                t = np.asarray(times, dtype=float)
                return int(((t >= cue + RESPONSE_WINDOW[0]) &
                            (t <= cue + RESPONSE_WINDOW[1])).sum())
            flag = pd.DataFrame({
                "trial": np.arange(len(cues)),
                "go_cue": cues,
                "n_lickometer": [_n_in_window(all_licks, c) for c in cues],
                "n_pose_only": [_n_in_window(pose_only_times, c) for c in cues],
                "n_confident": [_n_in_window(confident, c) for c in cues],
            })
            flagged = flag[(flag["n_lickometer"] == 0) & (flag["n_pose_only"] > 0)]
            flagged_hc = flag[(flag["n_lickometer"] == 0) & (flag["n_confident"] > 0)]
            print("\nB. per-trial flag, response window {}-{} s".format(*RESPONSE_WINDOW))
            print("   trials: {:,}".format(len(flag)))
            print("   no lickometer lick but a pose lick: {:,} ({:.1%} of trials)".format(
                len(flagged), len(flagged) / max(len(flag), 1)))
            print("   same, high-confidence only:         {:,} ({:.1%} of trials)".format(
                len(flagged_hc), len(flagged_hc) / max(len(flag), 1)))
            display(flagged.head(10))

## 9. Limitations

1. **A pose-only event is a candidate, not a missed lick.** Nothing here separates a lickometer
   failure from a pose false positive without watching the video. Every number labelled
   "candidate" carries the §5 precision as its error rate.
2. **The pixel threshold is a free parameter with no correct value** (§4). Every count in §7
   scales with it, so cross-study comparisons are meaningless unless the threshold is reported.
3. **Tracking coverage bounds everything.** Lickometer licks with no tracked tongue nearby can
   neither be confirmed nor denied; that fraction is printed in §4 and is a floor on the method's
   sensitivity.
4. **Not validated against the issue's session.** `behavior_856239_2026-07-24` has no pose output,
   so the specific case in issue #96 has not been reproduced. The rates in §7 come from a
   different cohort (44 sessions, 2024-05-31 to 2025-07-03) and may not transfer.
5. **One camera, one keypoint.** Detection rests on `tongue_tip_center` from the bottom camera.
   Occlusion, spout position drift within a session, and the fixed mean spout position all feed
   directly into the distance measure. Spout positions are session means here; if the spout moves
   mid-session, the distance trace is biased for part of it.
6. **No hand-scored validation set.** The precision numbers are all measured *against the
   lickometer*, which is the instrument under test. A genuine accuracy estimate needs a sample of
   events scored by eye from video — §6 extracts the clips for exactly that, but the scoring has
   not been done.

## 10. Export for the summary page

Writes the numbers behind every figure to `FIG_DIR/val_03_summary.json`. Set `SAVE_FIG = True`
in §1 and re-run to also write the figure PNGs alongside it — together those are what a
shareable write-up for issue #96 is built from.

In [ ]:
if not IS_CO:
    print("[skip] Export needs the analysis above.")
else:
    import json as _json

    def _grab(name):
        """Return a global if the cell that defines it ran, else None."""
        return globals().get(name)

    summary = {
        "session": EXAMPLE_SESSION,
        "params": {"pixel_threshold": PIXEL_THRESHOLD, "t_refractory": T_REFRACTORY,
                   "match_window": MATCH_WINDOW, "confidence": CONF},
        "example": {"duration_s": duration, "n_frames": int(len(tongue)),
                    "tracked_frac": float(tongue["x"].notna().mean()),
                    "n_lickometer": int(len(all_licks)),
                    "video_offset_s": VIDEO_OFFSET},
        "distance_at_licks_pctl": {
            str(q): float(np.nanpercentile(d_at_lick, q)) for q in [10, 25, 50, 75, 90]},
        "licks_without_tracking_frac": float(np.isnan(d_at_lick).mean()),
        "at_operating_point": {k: (float(ref[k]) if isinstance(ref[k], float) else int(ref[k]))
                               for k in ["n_matched", "n_pose_only", "n_lickometer_only",
                                         "precision", "recall"]},
    }

    sw = _grab("sweep_df")
    if sw is not None:
        summary["threshold_sweep"] = sw.drop(columns=[c for c in sw.columns
                                                      if c.endswith("_times")]).to_dict("records")
    sd = _grab("sessions_df")
    if sd is not None and len(sd):
        summary["sessions"] = sd.to_dict("records")
        summary["across_sessions"] = {
            "n": int(len(sd)),
            "pose_only_per_min_median": float(sd["pose_only_per_min"].median()),
            "pose_only_per_min_min": float(sd["pose_only_per_min"].min()),
            "pose_only_per_min_max": float(sd["pose_only_per_min"].max()),
            "precision_median": float(sd["precision"].median()),
            "tracked_frac_median": float(sd["tracked_frac"].median()),
        }
    cf = _grab("confident")
    if cf is not None:
        summary["metric_c"] = {"percentile": CONFIDENT_PCTL, "cutoff_px": float(cutoff),
                               "n_pose_only": int(len(pose_only_times)),
                               "n_confident": int(len(cf))}
    fl = _grab("flagged")
    if fl is not None:
        summary["metric_b"] = {"response_window": list(RESPONSE_WINDOW),
                               "n_trials": int(len(_grab("flag"))),
                               "n_flagged": int(len(fl)),
                               "n_flagged_high_conf": int(len(_grab("flagged_hc")))}

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / "val_03_summary.json"
    out.write_text(_json.dumps(summary, indent=2, default=float))
    print("wrote {}".format(out))
    print("sections captured: {}".format(", ".join(sorted(summary))))
    if not SAVE_FIG:
        print("\nSAVE_FIG is False - set it True in S1 and re-run to write the figure PNGs.")